# ⚙️ Topic 04: ML Pipeline Orchestration

## 1. Why Pipeline Orchestration?
In production, ML tasks cannot be run as manual Jupyter notebook execution steps. They require automated, modular Directed Acyclic Graphs (DAGs).

### Core Pipeline Principles:
1. **Modularity:** Separate steps for ingestion, preprocessing, training, and evaluation.
2. **Idempotency:** Re-running a step with identical inputs produces identical outputs.
3. **Quality Gatekeeping:** Automatically aborting deployment if evaluation metrics fall below baseline thresholds.

---

## 2. Hands-on: Modular ML Orchestrator with Quality Gates


In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

class MLPipeline:
    def __init__(self, accuracy_threshold=0.85):
        self.accuracy_threshold = accuracy_threshold
        self.state = {}

    def step_1_ingest_data(self):
        print("📥 Step 1: Ingesting Raw Data...")
        iris = load_iris()
        X = pd.DataFrame(iris.data, columns=iris.feature_names)
        y = iris.target
        self.state["X_raw"] = X
        self.state["y_raw"] = y
        return self

    def step_2_preprocess(self):
        print("🧹 Step 2: Preprocessing Features...")
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(self.state["X_raw"])
        X_train, X_test, y_train, y_test = train_test_split(
            X_scaled, self.state["y_raw"], test_size=0.2, random_state=42
        )
        self.state.update({
            "scaler": scaler,
            "X_train": X_train, "X_test": X_test,
            "y_train": y_train, "y_test": y_test
        })
        return self

    def step_3_train_model(self):
        print("🤖 Step 3: Training Logistic Regression Model...")
        model = LogisticRegression(max_iter=200)
        model.fit(self.state["X_train"], self.state["y_train"])
        self.state["model"] = model
        return self

    def step_4_evaluate_and_gatekeep(self):
        print("🛡️ Step 4: Evaluating Metric Gatekeeper...")
        preds = self.state["model"].predict(self.state["X_test"])
        acc = accuracy_score(self.state["y_test"], preds)
        print(f"   -> Evaluated Accuracy: {acc:.4f} (Required Threshold: {self.accuracy_threshold})")
        
        if acc >= self.accuracy_threshold:
            print("   ✅ QUALITY GATE PASSED! Model approved for staging/deployment.")
            self.state["gate_passed"] = True
        else:
            print("   ❌ QUALITY GATE FAILED! Aborting deployment pipeline.")
            self.state["gate_passed"] = False
        return self

    def run(self):
        print("======== STARTING ML PIPELINE DAG ========")
        self.step_1_ingest_data()\
            .step_2_preprocess()\
            .step_3_train_model()\
            .step_4_evaluate_and_gatekeep()
        print("======== PIPELINE COMPLETE ========\n")

pipeline = MLPipeline(accuracy_threshold=0.85)
pipeline.run()
